# Proposed validation — review before it runs

**What this measures:** The target `l0_deficit_from_k` now certifies the kept mechanism question on a TRAINED AbsTopK rather than a random-init encoder: the script drives the repo's own `LanguageModelSAETrainingRunner` twice with one identical config — control `topk` and test `abstopk`, k=64, d_sae=8192, 10M tokens of streaming pile on gpt2 `blocks.6.hook_resid_post`, both runs logged into ONE W&B group via the engine's `WANDB_*` env (the dashboards the PR template demands) — and on held-out gpt2 activations the new `!=0` counting must report exactly k=64 signed actives per token, with the guardrail `relu_topk_l0_predicate_delta` re-checking the `>0`→`!=0` no-op on the trained TopK control and a short-run MSE ratio plus a wandb-runs-logged sanity count reported as directional evidence (`kind: capability`, `baseline: none` — main cannot import AbsTopK, so both arms live inside this one run and no sentinel number is ever emitted).

**Target metric:** `l0_deficit_from_k`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at `benchmark/bench_abstopk_mechanism.py`, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

### Weights & Biases

This validation logs training runs to **Weights & Biases** so the dashboards the repository asks for exist. It needs your W&B API key (W&B → Settings → API keys). In Colab, store it as a secret named `WANDB_API_KEY`; elsewhere set the environment variable. The run goes to entity `smellslikeml` (your W&B username or a team you belong to) and project `remyx-validate-saelens` (the folder the runs are grouped under, created if it does not exist) — the same place the Remyx run logs to, so your run appears beside it. Change the two values below to log somewhere else.

In [ ]:
import os
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception:
        pass
os.environ.setdefault("WANDB_ENTITY", "smellslikeml")
os.environ.setdefault("WANDB_PROJECT", "remyx-validate-saelens")
os.environ.setdefault("WANDB_BASE_URL", "https://api.wandb.ai")
os.environ.setdefault("WANDB_TAGS", "remyx-validate,hand-run")
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY not set — W&B logging disabled for this run")
else:
    print(f"W&B: {os.environ['WANDB_ENTITY']}/{os.environ['WANDB_PROJECT']}")

## Execution context

The cells below are the script at `benchmark/bench_abstopk_mechanism.py`, unchanged. This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "benchmark/bench_abstopk_mechanism.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
# ruff: noqa: T201
"""AbsTopK vs TopK — a REAL training run, logged to Weights & Biases
(capability question, baseline: none — the control (topk) and, on the PR
head, the test (abstopk) both train inside this one run through the repo's
own LanguageModelSAETrainingRunner, the entry point the PR pre-registers in
VALIDATION.md; each arm lands in the engine's W&B run group under its own
run id, giving the PR template's "control and test group" dashboards. On a
pre-change checkout the TopK control still trains unchanged and the AbsTopK
arm degrades to explicit worst-case values via the defensive import — both
arms run end to end, nothing is faked).

Held constant (validation.yaml): one LanguageModelSAERunnerConfig per arm —
gpt2, blocks.6.hook_resid_post, monology/pile-uncopyrighted (streaming),
context_size 256, train_batch_size_tokens 4096, training_tokens 500k (a real
optimizer-driven run at the cpu tier the harness provisions; VALIDATION.md
scales the same protocol to 500M-1B tokens on GPU) — only the sae=
architecture differs; k=64, d_sae=8192, d_in=768. Mechanism metrics are read
off the TRAINED operators on held-out gpt2 activations (fixed text, first
500 token positions). The scorer never reads the wandb dashboard;
wandb_runs_logged only verifies the runs initialized AND finished.
"""
import argparse
import json
import os
import sys
from pathlib import Path

In [ ]:
REPO_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT))

import torch
from torch import nn
from transformer_lens import HookedTransformer

In [ ]:
# pre-existing, unchanged entry points — unguarded (a break here must raise)
from sae_lens import LanguageModelSAERunnerConfig, LanguageModelSAETrainingRunner
from sae_lens.saes import TopKTrainingSAEConfig

try:  # the only symbols this PR adds — absent on the baseline checkout
    from sae_lens.saes import AbsTopKTrainingSAEConfig

    HAS_ABSTOPK = True
except ImportError as exc:
    print(f"[abstopk-bench] AbsTopK not importable (baseline arm): {exc}", file=sys.stderr)
    HAS_ABSTOPK = False

In [ ]:
import wandb  # engine-provisioned (packages: [wandb]); WANDB_* env pre-set

SMOKE = os.environ.get("REMYX_SMOKE") == "1"
HOOK, K, D_IN, D_SAE = "blocks.6.hook_resid_post", 64, 768, 8192
TEXT = "The quick brown fox jumps over the lazy dog. " * 60  # ~600 tokens
N_POS = 128 if SMOKE else 500
TRAIN_TOKENS = 2 * 4096 if SMOKE else 500_000  # ~2 optimizer steps smoke / 500k real
DEVICE = "cpu"  # yaml tier: cpu — the harness provisions no GPU
BASE_RUN_ID = os.environ.get("WANDB_RUN_ID", "abstopk-bench")

In [ ]:
def _find_sae(obj, depth=0):
    """Locate the trained SAE module in whatever run() hands back (the return
    type is not part of the visible surface; this only walks attributes)."""
    if isinstance(obj, nn.Module) and hasattr(obj, "encode") and hasattr(obj, "decode"):
        return obj
    if depth > 3:
        return None
    if isinstance(obj, (tuple, list)):
        items = list(obj)
    elif isinstance(obj, dict):
        items = list(obj.values())
    elif hasattr(obj, "__dict__"):
        items = list(vars(obj).values())
    else:
        items = []
    for item in items:
        found = _find_sae(item, depth + 1)
        if found is not None:
            return found
    return None

In [ ]:
def train_arm(name, sae_cfg, seed):
    # split the engine's WANDB_RUN_ID per arm; entity/project/group come from env
    os.environ["WANDB_RUN_ID"] = f"{BASE_RUN_ID}-{name}"
    if wandb.run is not None:
        wandb.finish()
    torch.manual_seed(seed)
    cfg = LanguageModelSAERunnerConfig(  # the PR's own pre-registered entry point
        sae=sae_cfg,
        model_name="gpt2",
        hook_name=HOOK,
        dataset_path="monology/pile-uncopyrighted",
        streaming=True,
        training_tokens=TRAIN_TOKENS,
        train_batch_size_tokens=4096,
        context_size=256,
        device=DEVICE,
        log_to_wandb=True,
    )
    runner = LanguageModelSAETrainingRunner(cfg)
    sae = _find_sae(runner.run()) or _find_sae(runner)
    if sae is None:
        raise RuntimeError(f"arm {name}: no trained SAE returned by runner")
    print(f"[abstopk-bench] arm {name} trained on {TRAIN_TOKENS} tokens")
    return sae

In [ ]:
def encode_with_pre_acts(sae, x):
    """Run the real sae.encode; capture the encoder pre-activations with a
    forward hook on the unique d_in->d_sae Linear — no reimplementation of
    the (changed) operator or of the encoder."""
    lins = [
        m
        for m in sae.modules()
        if isinstance(m, nn.Linear) and m.in_features == D_IN and m.out_features == D_SAE
    ]
    if len(lins) != 1:
        raise RuntimeError(f"expected one {D_IN}->{D_SAE} encoder Linear, found {len(lins)}")
    box = {}
    handle = lins[0].register_forward_hook(
        lambda _m, _i, out: box.setdefault("pre", out.detach().float().clone())
    )
    try:
        acts = sae.encode(x).detach().float()
    finally:
        handle.remove()
    return acts, box["pre"]

In [ ]:
def measure(sae, x):
    sae = sae.to(DEVICE).eval()
    x = x.to(DEVICE)
    with torch.no_grad():
        acts, pre = encode_with_pre_acts(sae, x)
        a, p = acts.reshape(-1, D_SAE), pre.reshape(-1, D_SAE)
        nz = a != 0
        denom = nz.sum().clamp(min=1).float()
        top_idx = p.abs().topk(K, dim=-1).indices
        exp_mask = torch.zeros_like(nz)
        exp_mask.scatter_(1, top_idx, True)
        sign_exact = ((a[nz] * p[nz]) > 0).float().mean() if nz.any() else 0.0
        recon = sae.decode(acts)
        mse = ((x - recon) ** 2).mean().item()
        return {
            "l0": nz.sum(-1).float().mean().item(),  # new !=0 counting
            "l0_gt0": (a > 0).sum(-1).float().mean().item(),  # old >0 counting
            "neg_share": ((a < 0).sum().float() / denom).item(),
            "pos_ratio": ((a > 0).sum().float() / denom).item(),
            "selection_exact": (exp_mask == nz).all(-1).float().mean().item(),
            "sign_exact": float(sign_exact),
            "mse": mse,
        }

In [ ]:
def wandb_arm_finished(suffix):
    ent, proj = os.environ.get("WANDB_ENTITY"), os.environ.get("WANDB_PROJECT")
    if not (ent and proj):
        return False
    try:  # existence/completion probe only; metric values never come from the dashboard
        run = wandb.Api().run(f"{ent}/{proj}/{BASE_RUN_ID}-{suffix}")
        return run.state == "finished"
    except Exception as exc:  # noqa: BLE001 — a missing run IS the failure being measured
        print(f"[abstopk-bench] wandb lookup failed for -{suffix}: {exc}", file=sys.stderr)
        return False

In [ ]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default="")
    parser.add_argument("--ref", default="")
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()  # variant/ref accepted and ignored; seed fixes init

    model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
    _, cache = model.run_with_cache(TEXT, names_filter=HOOK)
    x = cache[HOOK][0, :N_POS].float()  # held-out activations, fixed text

    ctrl = train_arm("topk", TopKTrainingSAEConfig(k=K, d_in=D_IN, d_sae=D_SAE), args.seed)
    m_ctrl = measure(ctrl, x)
    print(f"[abstopk-bench] topk    l0={m_ctrl['l0']:.3f} mse={m_ctrl['mse']:.6f}")

    if HAS_ABSTOPK:
        test = train_arm(
            "abstopk", AbsTopKTrainingSAEConfig(k=K, d_in=D_IN, d_sae=D_SAE), args.seed
        )
        m_test = measure(test, x)
        print(f"[abstopk-bench] abstopk l0={m_test['l0']:.3f} mse={m_test['mse']:.6f}")
    else:
        print(
            "[abstopk-bench] baseline arm: AbsTopK absent — control trained, "
            "test-arm metrics degrade to worst case",
            file=sys.stderr,
        )
        m_test = None
    if wandb.run is not None:
        wandb.finish()

    # guardrail is real on BOTH arms: the TopK control exists pre-change too
    predicate_delta = float(abs(m_ctrl["l0_gt0"] - m_ctrl["l0"]))
    wandb_logged = wandb_arm_finished("topk") + (
        wandb_arm_finished("abstopk") if m_test is not None else False
    )

    if m_test is not None:
        # l0_deficit_from_k: exactly-k scatter of signed values -> deficit > 0
        # only if a selected pre-activation is exactly 0.0; sign-killing reads ~K/2.
        metrics = {
            "l0_deficit_from_k": float(K - m_test["l0"]),
            "relu_topk_l0_predicate_delta": predicate_delta,
            "bidirectionality_deviation": float(abs(m_test["neg_share"] - 0.5)),
            "old_predicate_undercount_ratio": float(m_test["pos_ratio"]),
            "magnitude_selection_exact": float(m_test["selection_exact"]),
            "sign_preservation_exact": float(m_test["sign_exact"]),
            "mse_ratio_abstopk_vs_topk": float(m_test["mse"] / max(m_ctrl["mse"], 1e-12)),
            "abstopk_registered": float(m_test is not None),
            "wandb_runs_logged": float(wandb_logged),
        }
    else:
        # baseline degradation (defensive import failed): worst-case value for
        # every test-arm metric; the control-derived guardrail stays measured
        metrics = {
            "l0_deficit_from_k": float(K),
            "relu_topk_l0_predicate_delta": predicate_delta,
            "bidirectionality_deviation": 0.5,
            "old_predicate_undercount_ratio": 1.0,
            "magnitude_selection_exact": 0.0,
            "sign_preservation_exact": 0.0,
            "mse_ratio_abstopk_vs_topk": 1e6,
            "abstopk_registered": 0.0,
            "wandb_runs_logged": float(wandb_logged),
        }
    print(json.dumps(metrics))

if __name__ == "__main__":
    main()

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "@remyx-ai validate — does the AbsTopK port select exactly the k largest-magnitude pre-activations per token with signs preserved (~half negative) on real gpt2 activations, and is the >0 -> !=0 eval-counting fix an exact no-op for ReLU variants?"
loop: {max_iterations: 8, fix_code: true}
requires: [wandb]
benchmarks:
  - name: abstopk-mechanism
    suite: "benchmark/bench_abstopk_mechanism.py"
    packages: [wandb]
    baseline: none
    scorer: l0_deficit_from_k
    policy: {guardrail_veto: true}
    # timeout derivation: 2 arms x ~122 steps (500k tokens / 4096 tokens-per-step)
    # x ~8-10 s/step on CPU ~= 35-40 min training + gpt2 download + pile-stream
    # warmup + held-out eval -> 5400 s ceiling for one job
    compute: {tier: cpu, timeout_s: 5400}
    metrics:
      - name: l0_deficit_from_k
        direction: min
        threshold: 0.0
        role: target
        bar: goal
        reads_as: "k minus mean per-token L0 under the new !=0 counting on the TRAINED AbsTopK; 0.0 = exactly k signed features survived (sign-killing reads ~k/2)"
      - name: relu_topk_l0_predicate_delta
        direction: min
        threshold: 0.5
        role: guardrail
        bar: goal
        reads_as: "absolute per-token L0 shift on the ReLU-family control from swapping >0 for !=0; correct arithmetic gives exactly 0.0, and the 0.5 bound (half an active feature per token) is a real gate — any wholesale predicate regression (>= 0 counting padding zeros, < 0 counting nothing) shifts L0 by >= 1.0, up to the full k=64, and fails it; measured on the trained TopK control, which exists identically on the baseline checkout, so the guardrail is baseline-achievable"
      - name: bidirectionality_deviation
        direction: min
        threshold: 0.15
        role: diagnostic
        reads_as: "|negative share of TRAINED active features - 0.5|; random-init 3-sigma band was 0.0084, trained features may specialize — 0.15 is a reading bar, not a gate"
      - name: old_predicate_undercount_ratio
        direction: min
        threshold: 0.9
        role: diagnostic
        reads_as: "old >0-counted L0 over new !=0-counted L0 on trained AbsTopK acts; must be <= 0.9 to demonstrate a real undercount — expected ~0.5 (ratio = 1 - negative share); 1.0 would mean no negative actives at all (sign-killing), i.e. the undercount the fix corrects does not exist"
      - name: magnitude_selection_exact
        direction: max
        threshold: 1.0
        role: diagnostic
        reads_as: "share of held-out tokens whose kept set equals exactly the k largest-magnitude pre-activations"
      - name: sign_preservation_exact
        direction: max
        threshold: 1.0
        role: diagnostic
        reads_as: "share of active features whose output sign matches their pre-activation sign"
      - name: mse_ratio_abstopk_vs_topk
        direction: min
        threshold: 1.0
        role: diagnostic
        bar: goal
        reads_as: "trained-AbsTopK MSE over trained-TopK MSE at matched k=64 on held-out gpt2 activations; paper claims <= 1 at scale — directional at 500k tokens"
      - name: abstopk_registered
        direction: max
        threshold: 1.0
        role: diagnostic
        bar: sanity
        reads_as: "1.0 when the repo runner resolves architecture abstopk end-to-end (the PR's registration block + AbsTopKTrainingSAE construction during a completed training arm); 0.0 on a checkout where the symbols are absent"
      - name: wandb_runs_logged
        direction: max
        threshold: 2.0
        role: diagnostic
        bar: sanity
        reads_as: "arms whose wandb run initialized and finished inside the engine's run group; 2.0 = control + test dashboards exist to link from the PR"
    held_constant:
      - "one LanguageModelSAERunnerConfig for both arms — model gpt2, hook blocks.6.hook_resid_post, dataset monology/pile-uncopyrighted (streaming), context_size 256, train_batch_size_tokens 4096, training_tokens 500k (a real optimizer-driven run at the cpu tier; the VALIDATION.md protocol scales the same config to 500M-1B tokens on GPU), device cpu — only the sae= architecture differs (topk control vs abstopk test)"
      - "k=64, d_sae=8192, d_in=768 — the PR pre-registered sweep point (VALIDATION.md)"
      - "held-out eval on a fixed text through the trained SAEs; identical batches fed to the AbsTopK test and the TopK control; mechanism metrics computed on float32 tensors from the trained weights"
      - "both arms log to one W&B group: the engine sets WANDB_API_KEY/ENTITY/PROJECT/RUN_GROUP, the script only splits WANDB_RUN_ID per arm (-topk / -abstopk)"
    avoid:
      - "the scorer never reads the wandb dashboard — metrics come from the script's JSON line; the dashboard is linked evidence for the PR template, not the instrument"
      - "run_evals is not driven directly (ActivationScaler/ActivationsStore constructor signatures are outside the visible context); training itself goes through the repo runner, and the changed !=0/>0 predicates are measured on the real trained-operator outputs"
      - "no hardcoded wandb entity or project — routing comes from the engine's env, never invented in the script"
      - "no wall-clock timing and no cuda — tier: cpu provisions no GPU; the fixed seed keeps the run reproducible up to streaming-buffer order, and the exact-k/sign/predicate metrics are drift-free"
      - "no unpinned live APIs: gpt2 weights and the pile stream are fetched by the repo's own loaders (model-name/dataset-path pinned, HF-cached); no CE/KL evals, no interpretability probing"
    report:
      headline: "A real W&B-logged training run (one group: TopK control vs AbsTopK test) trains both arms end-to-end through the repo runner on CPU; the mechanism holds on the trained AbsTopK — exactly-k magnitude selection, signs preserved, !=0 L0 counting negative actives — with short-run MSE parity directional at 500k tokens"
      findings:
        - "per-token L0 under the new !=0 counting falls {l0_deficit_from_k} short of k=64 on the trained AbsTopK (0.0 expected; ~32 would indicate sign-killing)"
        - "the >0 -> !=0 swap changes the trained TopK control's L0 by {relu_topk_l0_predicate_delta} (exactly 0.0 expected; guardrail bound 0.5, so any wholesale predicate regression reading >= 1.0 fails)"
        - "the negative share of trained active features sits {bidirectionality_deviation} from 0.5 (random-init 3-sigma band was 0.0084; trained features may specialize)"
        - "the old >0 counting reports {old_predicate_undercount_ratio} of the new L0 on trained AbsTopK acts (~0.5 = the corrected 2x undercount; bound 0.9, 1.0 = no negative actives)"
        - "{magnitude_selection_exact} of held-out tokens keep exactly the k largest-magnitude pre-activations"
        - "{sign_preservation_exact} of active features retain their pre-activation sign"
        - "trained AbsTopK reconstruction MSE is {mse_ratio_abstopk_vs_topk} times the trained TopK control's at matched k=64 (paper claims <= 1 at scale; 500k tokens is directional)"
        - "{abstopk_registered} registry wiring for architecture abstopk; {wandb_runs_logged} wandb runs (2.0 = control + test) landed in the engine's run group"
      establishes:
        - "AbsTopK trains end-to-end through the repo's own LanguageModelSAETrainingRunner and logs beside a matched TopK control into one W&B group — the control/test dashboards the PR template requires"
        - "on trained weights at the PR's pre-registered geometry, the operator still selects exactly the k largest-magnitude pre-activations with signs preserved, and the !=0 eval counting reports them"
        - "the >0 -> !=0 predicate swap remains an exact no-op for the trained ReLU-family (TopK) control"
      does_not_establish:
        - "full-scale sparsity-fidelity parity (500M-1B tokens on GPU, k in {16,64,256}, JumpReLU control) — the run the PR itself defers; the 500k-token MSE ratio is directional only"
        - "any interpretability claim: probing, steering, or single features encoding contrasting concepts"
        - "behaviour of the full run_evals loop through ActivationsStore — only its changed predicates are exercised on trained outputs"
        - "the latent issue that evals.py l1 is a signed sum and cancels toward 0 for AbsTopK (untouched by this PR)"
      not_measured:
        - "ce_loss_score, kl_div, explained_variance, dead-feature counts beyond the trainer's own wandb curves, throughput/wall-clock, feature-dashboard interpretability"
      caveat: "the paper's headline results (7 probing/steering tasks across 4 LLMs, matching supervised DiM) have no harness in this repo; this run supplies the template's control/test dashboards plus mechanism evidence, not the deferred parity frontier"
      next: "run the full VALIDATION.md protocol on GPU: shared activation cache, k in {16,64,256}, TopK and JumpReLU controls, 500M-1B training tokens, run_evals l0/mse/ce_loss_score plus dead-feature count on the same W&B project"
    provenance:
      suite: "synthesized (script drives the repo's own LanguageModelSAETrainingRunner entry point, pre-registered in protocol_doc:VALIDATION.md)"
      training_protocol: "protocol_doc:VALIDATION.md (gpt2, blocks.6.hook_resid_post, monology/pile-uncopyrighted streaming, train_batch_size_tokens 4096, context_size 256, k in {16,64,256} — single point k=64 at 500k tokens chosen as the affordable real run at the cpu tier)"
      wandb: "user_guidance (reviewer: real training run logging to W&B) + pr_template ('Please links to wandb dashboards with a control and test group')"
      l0_deficit_from_k: "inferred (claim arithmetic: exactly-k scatter of signed values; deficit > 0 only if a selected pre-activation is exactly 0.0)"
      relu_topk_l0_predicate_delta: "inferred (claim arithmetic: (t>0) == (t!=0) elementwise for t >= 0 gives exactly 0.0; bound 0.5 = half an active feature per token, chosen between exact correctness (0.0) and the smallest systematic regression (1 full feature/token, up to k=64 for a wrong operator/tensor) so the guardrail can actually fail; measured on the TopK control, which trains identically on the baseline checkout, so the bound is baseline-achievable)"
      old_predicate_undercount_ratio: "inferred (ratio = positive actives / all actives = 1 - negative share; expected ~0.5; bound 0.9 fails only when neg_share < 0.10, i.e. essentially no undercount for the fix to correct)"
      bidirectionality_deviation: "inferred (random-init 3-sigma band was 0.008385 at 32000 signs; trained features may specialize directionally, so the reading bar is relaxed to 0.15 — diagnostic, non-gating)"
      mse_ratio_abstopk_vs_topk: "pr_template (MSE Loss performance row) + paper claim (arXiv:2510.00404: AbsTopK matches or beats TopK at matched sparsity); 500k-token single-k run is directional evidence"
      wandb_runs_logged: "user_guidance"
      baseline_fallback: "inferred (kind: capability, baseline: none — the script still runs on a pre-change checkout without crashing: the TopK control trains unchanged through the pre-existing runner, the guardrail is measured on it for real, and every test-arm metric degrades to its worst case via the defensive import; no fabricated test-arm number is ever emitted)"
      compute: "user_guidance (the ask is a real training run, honored end-to-end through the repo runner) at tier cpu — the harness provisions no GPU, so the token budget is 500k; timeout derived from 2 x ~122 steps x ~8-10 s/step + gpt2 download + pile warmup + held-out eval"
      held_constant: "protocol_doc:VALIDATION.md"
logging:
  wandb:
    entity: smellslikeml    # from your W&B connection
    project: remyx-validate-saelens    # default
```